#### Notebook for running React experiments

In [ ]:
import sys, os

sys.path.append("..")
root = "../root/"

In [ ]:
from dotenv import load_dotenv

load_dotenv()


In [ ]:
import joblib
from util import summarize_react_trial, log_react_trial, save_agents
from agents import ReactReflectAgent, ReactAgent, ReflexionStrategy

#### Load the HotpotQA Sample

In [ ]:
hotpot = joblib.load("../data/hotpot-qa-distractor-sample.joblib").reset_index(
    drop=True
)
hotpot = hotpot[:2]

hotpot

#### Define the Reflexion Strategy

In [ ]:
print(ReflexionStrategy.__doc__)

In [ ]:
strategy: ReflexionStrategy = ReflexionStrategy.REFLEXION

#### Initialize a React Agent for each question

In [ ]:
agent_cls = ReactReflectAgent if strategy != ReflexionStrategy.NONE else ReactAgent
agents = [agent_cls(row["question"], row["answer"]) for _, row in hotpot.iterrows()]

#### Run `n` trials

In [ ]:
n = 2
trial = 0
log = ""

In [ ]:
for i in range(n):
    for agent in [a for a in agents if not a.is_correct()]:
        if strategy != ReflexionStrategy.NONE:
            agent.run(reflect_strategy=strategy)
        else:
            agent.run()
        print(f"Answer: {agent.key}\n")
    trial += 1
    log += log_react_trial(agents, trial)
    correct, incorrect, halted = summarize_react_trial(agents)
    print(
        f"Finished Trial {trial}, Correct: {len(correct)}, Incorrect: {len(incorrect)}, Halted: {len(halted)}"
    )
    print("*" * 20)

#### Save the result log

In [ ]:
with open(
    os.path.join(
        root, "ReAct", strategy.value, f"{len(agents)}_questions_{trial}_trials.txt"
    ),
    "w",
) as f:
    f.write(log)
save_agents(agents, os.path.join("ReAct", strategy.value, "agents"))